In [2]:
# !wget -P ../data https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [3]:
# read the text

with open("../data/input.txt", "r") as file: 
    text = file.read()

### Numerical Embeddings

This converts letters to numerical values for embeddings such that a computer can use to index into a table or something. You need this because, later on when table embeddings come into picture, you need a way to get the embedding values ("relationships" to other characters/tokens) so you can use the numerical values produced to index. 

And then `itos` actually allows you to see which words something would reference like converting, for instance, 65 --> "h". 

In [4]:
# convert text into numerical embeddings

chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


### Using `torch`

Converting data into a tensor allows for direct operations to be produced such as `view`, `reshape`, etc. Then that would allow you to perform further operations and pass into NNs, the actual backbone of the LLM. 

In [ ]:
import torch
data = torch.tensor(encode(text))

In [ ]:
train = data[:int(0.9 * len(data))]
test = data[int(0.9 * len(data)) + 1:]

### Representing "attention"

In this case, we are saying that the sample `y` is one token ahead of `x`. In this case, we are saying that the tokens leading up to the current point (represented by indexing in `x`) allow us to predict the next token which is `y[t]`. 

In [ ]:
block_size = 32
sample = train[:block_size + 1]

x = train[:block_size]
y = train[1:block_size + 1]

for t in range(block_size):
        context = x[:t + 1]
        output = y[t]
        print(f"Context: {context} is for {output}")

Context: tensor([18]) is for 47
Context: tensor([18, 47]) is for 56
Context: tensor([18, 47, 56]) is for 57
Context: tensor([18, 47, 56, 57]) is for 58
Context: tensor([18, 47, 56, 57, 58]) is for 1
Context: tensor([18, 47, 56, 57, 58,  1]) is for 15
Context: tensor([18, 47, 56, 57, 58,  1, 15]) is for 47
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47]) is for 58
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]) is for 47
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]) is for 64
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64]) is for 43
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43]) is for 52
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52]) is for 10
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10]) is for 0
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0]) is for 14
Context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14]) is for 43
Context: t

### Batches

We are creating batches of random indexes that represent starting points. Basically, we take the current index + `block_size` and then grab that as one batch corresponding to the provided index. We do the same for `y` except we are one position ahead because we are conveying that we are aiming to predict the next token, represented by `block_size + 1`. 

Then we go through every batch produced, and then we index to represent the same way that we did before. What the point of this cell is that we are representing in parallel processing of multiple batches at once for training. This way, we are processing `batch_size` token sequences of `block_size` length at the same time (in practical applications with `batch_size` threads on a consumer GPU) and then getting the relationship for each.

In [ ]:
batch_size = 4

def get_batch(split): 
    data = train if split == "train" else test
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y
    
xb, yb = get_batch("train")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"When input is {context.tolist()} the target is {target}")
        

When input is [54] the target is 43
When input is [54, 43] the target is 39
When input is [54, 43, 39] the target is 49
When input is [54, 43, 39, 49] the target is 6
When input is [54, 43, 39, 49, 6] the target is 1
When input is [54, 43, 39, 49, 6, 1] the target is 52
When input is [54, 43, 39, 49, 6, 1, 52] the target is 53
When input is [54, 43, 39, 49, 6, 1, 52, 53] the target is 58
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58] the target is 1
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1] the target is 44
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44] the target is 53
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44, 53] the target is 56
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44, 53, 56] the target is 1
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44, 53, 56, 1] the target is 53
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44, 53, 56, 1, 53] the target is 59
When input is [54, 43, 39, 49, 6, 1, 52, 53, 58, 1, 44, 53, 56, 1, 53, 59

### Table Embeddings

A table embedding, what was mentioned before, is just a grid representation of learned relationship so far between tokens -- it represents the correlation between tokens when we want to know what is most likely to go after token X. The way to find that is to index the `Xth` row in the table embedding through `nn.Embedding` and that returns the relationships between all other tokens in the set of available tokens. 

`logits` represents the raw values of passing through the neural network before filtering. We use `view` applied on both tensors because `F.cross_entropy` expects the tensors in forms of `B x C x T` while we currently have it in `B x T x C`. Therefore, we apply `view` to just temporarily get a different shape WITHOUT copying and rearranging the underlying memory shape (`reshape` actually changes the shape, which we don't want because that would make the tensor). 

`generate` should take the (B, T) tensor given and extend it by one -- it should extend in the time dimension for each of the batches. You do this by getting the `logits` out of passing the new index into the `forward` function, and then just concatenating the new index on to the pre-existing argument that was passed in. 

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table_embedding = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, index, targets=None) -> tuple[torch.Tensor, torch.Tensor]:
        logits = self.table_embedding(index) # (B, T, C)
        # loss = F.cross_entropy(logits, targets) # this won't work
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_tokens_generate): # index is (B, T) tensor format
        # first, get the actual raw values or `logits`
        for _ in range(max_tokens_generate):
            logits, _ = self(index)
            logits = logits[:, -1, :] # this will take just the last timestep for all batches with all vocab size --> (B, C)
            # convert into probabilities from embeddings; we don't want raw logits, we want the converted probabilities or likelihood of what is the next character
            probs = F.softmax(logits, dim=-1) # still (B, C)
            # then sample the next token
            next_idx = torch.multinomial(probs, 1) # then this becomes just (B, 1)
            index = torch.cat((index, next_idx), dim=1)
        return index
        
        
m = BigramLanguageModel(65)
logits, loss = m(xb, yb)
preds = m.generate(index=torch.zeros((1,1), dtype=torch.int), max_tokens_generate=100)
preds = preds.tolist()
print(decode(xb[1].tolist()))
decode(preds[0])


TH:
True, when avoided grace mak


'\nSKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp\nwnYWmnxKWWev-tDqXErVKLgJ'

### Training the Bi-Gram Language Model

Use an `AdamW` optimizer, an optimizer that is much faster and more advanced, making it popular into today's world. 

Then, constantly sample batches that can be used for training, and then pass in these batches to the model. Then, obtain the `loss` and `logits`, and then make sure to zero out all the gradients, not to mess with the next training run. 

Note: Convert to Python script for portability and usage!

Then, create an `estimate_loss` function that essentially will average out the gradient updates over the last couple of training periods. For instance, you should use some kind of structure to maintain the gradients measured at every point, and then it should average out the loss with the current gradients to find the new gradients. 

In [ ]:
"""
we can train by just run for some amount epochs in a training loop, then pass in a batch into the forward pass. then take
the loss and give that to the optimizer for step and then zero the gradients out to prepare for the next training run. 
"""

m = BigramLanguageModel(65)
optimizer = torch.optim.AdamW(params=m.parameters(), lr=0.001)
for _ in range(10000):
    xb, yb = get_batch("train")
    logits, loss = m.forward(xb, yb)
    
    optimizer.zero_grad()
    
    loss.backward()
    optimizer.step()
    
    print(loss.item())


4.54642915725708
4.513073921203613
4.528194427490234
4.545904159545898
4.543483734130859
4.633749485015869
4.416658401489258
4.627524375915527
4.675907135009766
4.426915168762207
4.473148822784424
4.541120529174805
4.730560302734375
4.728875637054443
4.564064025878906
4.614599227905273
4.647702217102051
4.657293319702148
4.675101280212402
4.507514953613281
4.57080602645874
4.617286682128906
4.626962184906006
4.51264762878418
4.54918098449707
4.644956111907959
4.469387054443359
4.438172340393066
4.609301567077637
4.440680980682373
4.592968940734863
4.574361324310303
4.4690375328063965
4.643896102905273
4.548647880554199
4.650306224822998
4.5526509284973145
4.600003719329834
4.669330596923828
4.554083347320557
4.5402302742004395
4.539078712463379
4.653897762298584
4.6269354820251465
4.494202136993408
4.495353698730469
4.486288547515869
4.52490234375
4.585784435272217
4.480515003204346
4.559354782104492
4.484078884124756
4.47373104095459
4.5610151290893555
4.546398162841797
4.503785610198

In [ ]:
xb, yb = get_batch("val")

preds = m.generate(index=xb, max_tokens_generate=100)
preds = preds.tolist()
for i in range(len(xb)):
    print(f"Prompt: {decode(xb[i].tolist())}\n")
    print(f"Response: {decode(preds[i])}")
    print("**************************")

Prompt: r boy;
'Twere good he were schoo

Response: r boy;
'Twere good he were schoo tho; be ch'sthase't IO:
Pllifthrdore;
M: cod ho to th3Q;
HAtine; preaven bmuns, 's elily n ceve-
we
**************************
Prompt: ave, who never
Yields us kind an

Response: ave, who never
Yields us kind an Sure pyoV! dacheemo TO: ags stDYO tr y hak we frer;
pre wrsay ymandsave o re he PElu s w
Pvofreglit
**************************
Prompt:  they are louder than
the weathe

Response:  they are louder than
the weathe, wof ngend che at mat t ow silot c.
Wheso ysdel mp;
cereruld
A:
Antthas,

Whe shie so l MERo tlf, w
**************************
Prompt: ! 'tis a world to see,
How tame,

Response: ! 'tis a world to see,
How tame, ntmor Is
wis sthaqlithilone ncjquct LWeghece, bulaindes pag:
Mouswanduno!
KIUE:
EOROLO:
Nonthewrvis
**************************


### Introducing ATTENTION!

Let's say we have a structure of shape (B,T,C) (ex. B,T,C = 4,8,2) -- which is pretty simple for actual attention, but it will work for now. We want the `t`-th token in the current batch to have attention to the last `t` tokens in some weightage like an average of the last `t` embeddings including the current (which again, is simple, but it will work for now.) We don't want the `t`-token to have a relationship with the future tokens, establishing a *causal* relationship for attention. 

In [ ]:
B,T,C = 4,8,2
x = torch.randn((B,T,C))

xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0)


We can perform the same operation with torch without having to run a `for` loop. We can use matrix multiplication. Basically, multiplying an lower-triangular identity matrix by the target matrix would give the sum across the rows. Then, we can just divide the original identity matrix first by the sum across the rows. 

We use `keepdim` to make sure to maintain the dimensions and make sure broadcasting is easier. 

In [ ]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(dim=1, keepdim=True)
b = torch.tensor([[1, 2], [3, 4], [5, 6]], dtype=torch.float)
c = a @ b
c

tensor([[1., 2.],
        [2., 3.],
        [3., 4.]])

So, in this case, we want to do the same thing as this 

``` B,T,C = 4,8,2
x = torch.randn((B,T,C))

xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0)
```

but with matrix multiplication instead and not `for` loops. So what we should do is multiply `x` by `b = torch.ones(B, T, C) / torch.sum(b, dim=1, keepdim=True)`. so `wei` would be `b` in this case and we would set it to the weights that represents how much each of the previous `k` values should contribute to the current embedding value, which is an average.

In [ ]:
wei = torch.tril(torch.ones(size=(T,T), dtype=torch.float)) # (T,T)
wei = wei / wei.sum(dim=1, keepdim=True) # still (T,T)
out = wei @ x # this is (T,T) * (B,T,C) --> DIMENSION MISMATCH! torch broadcasts such that (T,T) -> (B,T,T). (B,T,T) * (B,T,C) --> (B -- stays the same, T,C). 
# the last two dimensions always gets treated as the matrix dimensions that get multiplied as traditional matrices
out

tensor([[[-0.8799,  0.0057],
         [-0.5101,  0.6696],
         [-0.1058,  0.5458],
         [-0.3439,  0.1527],
         [-0.2363,  0.0020],
         [-0.0742,  0.0450],
         [-0.2266, -0.0605],
         [-0.2819, -0.0470]],

        [[ 0.0923, -1.2496],
         [ 0.6816,  0.3932],
         [-0.0420, -0.0433],
         [-0.3850,  0.3032],
         [-0.6042,  0.2897],
         [-0.4827,  0.4809],
         [-0.3760,  0.4501],
         [-0.1434,  0.3633]],

        [[-0.0669,  0.8775],
         [-0.5097,  0.4426],
         [-0.3507,  0.2316],
         [-0.0976, -0.0383],
         [-0.1618, -0.0488],
         [-0.1773, -0.0725],
         [-0.1612, -0.0499],
         [-0.1503,  0.1264]],

        [[-0.4793, -0.8862],
         [-0.7223, -0.4696],
         [-0.2821, -0.1389],
         [-0.3855, -0.3105],
         [-0.1943, -0.1728],
         [-0.4872, -0.4183],
         [-0.2934, -0.2689],
         [-0.0873, -0.2254]]])

### Attention: Softmax Version

You can also do the following, which is what will be used for self-attention, which is using `softmax` as a way to transform the lower triangular matrix or the actual weights that we use for attention. The important part is that we represent that tokens that should have no "affinity" or connection with the current token (future tokens) with float(-inf) instead of zeros (but after we transform using `softmax`, they turn into zeros so...) then multiply those weights with the original input tensor. 

In [ ]:
tril = torch.tril(torch.ones(T,T, dtype=torch.float))
wei = torch.zeros(T,T, dtype=torch.float)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow3, xbow)

True

### Positional Embeddings

So far, we have been working with embeddings and predictions that work slightly, but don't necessarily make sense because of one of two flaws: lack of positional importance. We can solve this using positional embeddings, where we assign each token a specific positional value on top of the current identity value that we already have. Specifically, we can use another table embedding that has dimensions `BLOCK_SIZE` by `N_EMBD`, and then adds this `pos_emb` to the `tok_emb` value, which is the then passed into the `lm_head`. This means that, in theory, it should use both positional embedding as well as token embeddings to determine that next best character, not just `pos_emb`. But, with a BiGram model, it wouldn't matter because we don't use other tokens, just the current token, to predict the next token. 

### MOST IMPORTANT PART: SELF-ATTENTION!

Self-attention allows you to specifically place data-dependent importance on different tokens from the perspective of the current token, not just have equal weightage or just use the previous token. 

Starting from now, each token will emit two vectors: a Q, query, and a K, for key. The Q vector roughly equates to "What am I looking for?" and the K vector roughly equates to "What do I contain?" For the current token, we can perform a dot product between the current token's Q vector and the other tokens' (previous tokens!) K vectors -- the one provides a higher affinity would be more representative as to what tokens matters most to predicting the next token.     

So, technical details: K and Q are two vectors that are produced by linear layers (`nn.Linear`) that takes in C values and outputs a `HEAD` dimension of something (typically 16). This means that the questions of "What am I looking for?" and "What do I contain?" are represented with embeddings of 16 values. 

Then, we take the dot product. Of course, we can't just straight multiply a K: (B, T, 16) by a Q: (B, T, 16), so we transpose K = (B, 16, T). Then, we take the dot product which means it becomes (B, T, 16) * (B, 16, T) which then turns into (B, T, T). What does this represent? In this case, there is a TxT table embedding essentially representing the affinity between each token, not just equal weightage. In that case, it determines what token/s should contribute the most out of the T tokens available within that batch. 

But one more thing: you have the **value** as well. Think of the private information being the raw index `x`, which contains the true indices of the tokens. Whenever a token is attending to other past tokens, we don't necessarily want to reveal this information directly. So instead, we have a V vector as well that also is a linear layer similar to K and Q with the same dimensions that shows that actual value that represents the "value" of that token or whatever the token contributes when shown or selected for inspection. For instance, say a token gets selected with relation to "France" based on certain characteristics revealed in K -- you don't want to reveal just K, you want to reveal the important bits that might or might not be K, which is learned and exposed through V. **You multiply the weights calculated in the (B, T, T) matrix from the dot product, and then multiply those by V to get the output**.

**Attention is a communication mechanism**. We can establish connections between tokens with establish weightages that kind of resemble a graph with directed edges. So for instance, token 1 just has itself pointing, but token 2 has token 1 and token 2 pointing to token 2. With this, there is no notion of space -- we are just connecting vectors, but none of these vectors know where they are relative to other tokens. That is why we have positional embedding to add some notion of position to each of these vectors, that way, a vector in index 5 is different from the same embedding in index 1. 

In [ ]:
torch.set_printoptions(sci_mode=False)
torch.manual_seed(1447)
HEAD = 16
k = nn.Linear(C, HEAD, bias=False)
q = nn.Linear(C, HEAD, bias=False)
v = nn.Linear(C, HEAD)

key = k(x) # (B, T, VOCAB_SIZE) -> (B, T, 16)
query = q(x) # (B, T, VOCAB_SIZE) -> (B, T, 16)

wei = query @ torch.transpose(key, -2, -1) * HEAD ** -0.5
tril = torch.tril(torch.ones(T,T,dtype=torch.float))

# wei = torch.zeros(T,T, dtype=torch.float)
wei = wei.masked_fill(tril == 0, float('-inf')) 
wei = F.softmax(wei, dim=-1) # (B,T,T) | you turn qk values into probabilities

value = v(x) # (B,T,16)  
out = wei @ value

wei # [B, T, 16] --> [B, T, 16] but with information including previous tokens

NameError: name 'torch' is not defined

There is the **encoder** and the **decoder**. The decoder is what is above in a nutshell, but the encoder is a situation where you can have all tokens attend to each other, for instance, in sentiment prediction of a sentence -- you are not trying to generate a next token or something like that.

**Self-attention** is where you have it so that QKV all come from the same source of data (in this case, `x`.) But then **cross-attention** is where you could have, for instance, V come from `x` but QK would come from `y`. 

One more thing is **scaled** self-attention where we divide the head dimension (in this case, 16.) This is because as the differences between positive raw embeddings and negative raw embeddings become larger, the `softmax` operation applied to the vectors become more like "one-hot encoded" vectors and then it starts converging to one specific max value compared to other tokens. So dividing by this scale allows for variance to be reduced, and then the probabilities become more dispersed. 

In [ ]:
key.var()
query.var()
wei.var()

tensor(0.0280, grad_fn=<VarBackward0>)

### Pre-Training, Post-Training, RLHF, and more

What is present in `v3.py` is the **pre-training** stage. Essentially, we are just training on some dataset, but at the end of pre-training, it just becomes a document completer, not a question-answer type of machine (if you fed in a question, it could just continue to generate more questions, for instance) -- it isn't capable of that right now after execution. 

However, **fine-tuning** is what "aligns" the LLM into the right area or boundary. You can give it structured question-answers sequences that resemble what ChatGPT or Claude SHOULD do. There are 3 steps to this:

1. We train the **SFT model**, which is the supervised fine-tuned model. We pull questions, a human writes down the ideal answer, and then train the SFT model to try to replicate the ideal answer. 
2. We train the **reward model**. Based on different responses the SFT model can give, a human ranks the responses. Multiple of these rankings allows for the reward model to dictate and "rank" responses through assignment of rewards, like assigning one response `reward = 3.7`, but another `reward = -1.2`. 
3. Finally, we train the **policy model** whose base is off the SFT model. Then, over time, the policy model generates responses, and then the reward model will assign rewards. Based on the reward given, you can adjust the response that you want to give next time. 